# 08. 대규모 데이터 수집/전처리/품질 설계

대규모 실제 데이터 경험은 모델 코드보다 데이터 계약에서 드러납니다.
여기서는 멀티모달 샘플 스키마, 회사명 마스킹, split 누수 검사, 품질 점수를 만듭니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
raw = pd.DataFrame([
    ("session01", "audio001.wav", "doc001.png", "회사명: Alpha / invoice A17", "clean", 0.95),
    ("session01", "audio002.wav", "doc001.png", "same document repeated", "noise", 0.71),
    ("session02", "audio003.wav", "doc002.png", "소속: Beta / alarm log", "alarm", 0.88),
    ("session03", "audio004.wav", "doc003.png", "client: Gamma / handwritten note", "blur", 0.63),
    ("session04", "audio005.wav", "doc004.png", "public manual page", "clean", 0.92),
], columns=["session_id", "audio_path", "image_path", "raw_text", "condition", "source_quality"])

import re
def mask_sensitive(text):
    text = re.sub(r"(회사명|소속|client)\s*[:：]\s*[^/\n]+", r"\1: xxx ", text, flags=re.IGNORECASE)
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "xxx@xxx", text)
    return text

raw["masked_text"] = raw["raw_text"].map(mask_sensitive)
raw["has_audio"] = raw.audio_path.str.endswith(".wav")
raw["has_image"] = raw.image_path.str.endswith(".png")
raw["quality_bucket"] = pd.cut(raw.source_quality, bins=[0, 0.7, 0.85, 1.0], labels=["low", "mid", "high"])
raw


In [ ]:
def assign_split(session_id):
    # 세션 단위 split으로 같은 문서/녹음 조건이 train/test에 동시에 들어가는 누수를 줄입니다.
    h = sum(ord(c) for c in session_id) % 10
    if h < 7:
        return "train"
    if h < 9:
        return "valid"
    return "test"

raw["split"] = raw["session_id"].map(assign_split)
leakage = raw.groupby("session_id")["split"].nunique().reset_index(name="n_splits")
assert leakage["n_splits"].max() == 1, "같은 session_id가 여러 split에 섞였습니다."

data_card = {
    "task": "multimodal field document QA",
    "modalities": ["speech", "environment sound", "document image", "text"],
    "sensitive_fields": ["company_name", "email", "client_name"],
    "masking_policy": "본문/로그/보고서에는 회사명과 고객명을 xxx로 표기",
    "split_policy": "session_id 단위 분리",
    "known_risks": ["ASR noise bias", "OCR low contrast", "layout leakage", "language imbalance"],
}
save_json("data_card.json", data_card)
raw.to_csv(ARTIFACTS / "multimodal_dataset_manifest.csv", index=False, encoding="utf-8-sig")
raw


## 확장 과제

- 언어별, 소음 조건별, 문서 유형별 샘플 수를 집계합니다.
- 품질 낮은 샘플을 버리는 실험과 보정해서 쓰는 실험을 비교합니다.
- 사내/고객 이름은 논문 표, 로그, 파일명에서 모두 `xxx`로 남도록 테스트를 추가합니다.
